# Hybrid LSTM Model for Credit Risk Prediction

This notebook implements a hybrid LSTM model that combines:
- **Sequential features**: Transaction sequences processed by LSTM layers
- **Static features**: User-level aggregated features (loan stats, balances, etc.)

The architecture fuses both inputs using a concatenate layer before the output.

## 1. Setup

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from seqcredit_model.credit_model import (
    CreditRiskDataLoader, HybridLSTMModel, ModelEvaluator, set_random_seeds
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

set_random_seeds(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('Setup complete.')

## 2. Data Preparation

In [ ]:
loader = CreditRiskDataLoader(
    features_path=os.path.join(project_root, 'data/user_features.csv'),
    summaries_path=os.path.join(project_root, 'data/user_labels.csv'),
    transactions_dir=os.path.join(project_root, 'data/user_transactions'),
)

static_data = loader.prepare_static_splits()

print(f"Static features: {len(static_data['feature_names'])}")
print(f"Training samples: {len(static_data['y_train'])}")
print(f"Test samples: {len(static_data['y_test'])}")

In [ ]:
seq_data = loader.load_sequences(
    max_seq_len=100,
    cache_path=os.path.join(project_root, 'data/lstm_sequences.npz'),
)

print(f"Sequence shape: {seq_data['X_train_seq'].shape}")
print(f"Sequence features: {len(seq_data['feature_names'])}")
print(f"Train default rate: {seq_data['y_train'].mean():.4f}")
print(f"Test default rate: {seq_data['y_test'].mean():.4f}")

## 3. Prepare Hybrid Data

The hybrid model requires:
- `X_train_seq`, `X_train_static`
- `X_test_seq`, `X_test_static`

We need to align the sequence and static data by user.

In [ ]:
from sklearn.preprocessing import StandardScaler

df_features = pd.read_csv(os.path.join(project_root, 'data/user_features.csv'))
df_summaries = pd.read_csv(os.path.join(project_root, 'data/user_labels.csv'))
df = df_features.merge(df_summaries, on='user_id', how='inner')
df = df[df['credit_risk_label'] != -1].copy()
df['default'] = (df['credit_risk_label'] == 2).astype(int)
df = loader._engineer_loan_features(df)

drop_cols = [
    'user_id', 'credit_risk_label', 'credit_archetype', 'default', 'gen_txn_count'
]
feature_cols = [c for c in df.columns if c not in drop_cols]
X_all = df[feature_cols].copy()
y_all = df['default'].copy()
user_ids = df['user_id'].values

static_scaler = StandardScaler()
X_all_scaled = static_scaler.fit_transform(X_all)

# Reuse the same train/test split from loader.prepare_static_splits()
# to ensure the hybrid model is evaluated on the same users as the static models.
train_user_ids = loader._train_user_ids
test_user_ids = loader._test_user_ids

print(f"Total borrowers: {len(user_ids)}")
print(f"Train: {len(train_user_ids)}, Test: {len(test_user_ids)}")

In [ ]:
from seqcredit_model.feature_engineering import TemporalTransactionFeatureEngineer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from pathlib import Path

transactions_path = Path(project_root) / 'data' / 'user_transactions'
engineer = TemporalTransactionFeatureEngineer()

def build_hybrid_data(user_id_set, user_to_idx, X_static_all, y_all, user_ids, max_seq_len=100):
    """Build aligned sequence and static data for a set of user IDs."""
    seq_list = []
    static_list = []
    label_list = []
    
    for uid in user_id_set:
        if uid not in user_to_idx:
            continue
        idx = user_to_idx[uid]
        
        filepath = transactions_path / f'{uid}.csv'
        if not filepath.exists():
            continue
        
        try:
            df_user = pd.read_csv(filepath)
            if df_user.empty or len(df_user) < 2:
                continue
            
            df_user['is_loan_disbursement'] = (df_user['TRANS. TYPE'] == 'CREDIT').astype(int)
            df_user['is_loan_repayment'] = (df_user['TRANS. TYPE'] == 'LOAN_REPAYMENT').astype(int)
            
            df_features = engineer.extract_all_features(df_user)
            available_cols = [c for c in seq_data['feature_names'] if c in df_features.columns]
            seq = df_features[available_cols].values.astype(np.float32)
            seq = np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0)
            
            seq_list.append(seq)
            static_list.append(X_static_all[idx])
            label_list.append(y_all[idx])
        except Exception:
            continue
    
    X_seq = pad_sequences(seq_list, maxlen=max_seq_len, padding='pre', truncating='pre', dtype='float32', value=0.0)
    X_static = np.array(static_list)
    y = np.array(label_list)
    
    return X_seq, X_static, y

user_to_idx = {uid: i for i, uid in enumerate(user_ids)}

print("Building training data...")
X_train_seq, X_train_static, y_train = build_hybrid_data(
    train_user_ids, user_to_idx, X_all_scaled, y_all.values, user_ids
)

print("Building test data...")
X_test_seq, X_test_static, y_test = build_hybrid_data(
    test_user_ids, user_to_idx, X_all_scaled, y_all.values, user_ids
)

print(f"Train: seq {X_train_seq.shape}, static {X_train_static.shape}, y {y_train.shape}")
print(f"Test: seq {X_test_seq.shape}, static {X_test_static.shape}, y {y_test.shape}")

## 4. Model Training

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes.astype(int), weights))
print(f'Class weights: {class_weights}')

hybrid_model = HybridLSTMModel(
    lstm_units_1=32,
    lstm_units_2=16,
    dense_units=16,
    dropout_rate=0.4,
    learning_rate=0.0005,
)
seq_shape = (X_train_seq.shape[1], X_train_seq.shape[2])
static_dim = X_train_static.shape[1]
hybrid_model.build_model(seq_shape, static_dim)
hybrid_model.model.summary()

In [ ]:
history = hybrid_model.fit(
    X_train_seq, X_train_static, y_train,
    epochs=100, batch_size=32,
    class_weight=class_weights,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['auc'], label='Train AUC')
ax2.plot(history.history['val_auc'], label='Val AUC')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('AUC')
ax2.set_title('Training AUC')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

hybrid_proba = hybrid_model.predict_proba(X_test_seq, X_test_static)

print('Hybrid LSTM - Test Set Performance:')
print(f'  AUC-ROC: {roc_auc_score(y_test, hybrid_proba):.4f}')
print(f'  AUC-PR: {average_precision_score(y_test, hybrid_proba):.4f}')

print('\nClassification Report:')
print(classification_report(y_test, (hybrid_proba >= 0.5).astype(int), target_names=['Non-Default', 'Default']))

## 6. Model Comparison

In [ ]:
evaluator = ModelEvaluator(y_test)
evaluator.add_model('Hybrid LSTM', hybrid_proba)

comparison = evaluator.get_comparison_table()
print('\nModel Comparison (Test Set):')
print(comparison.round(4).to_string())

In [ ]:
evaluator.plot_roc_curves()
plt.show()

In [ ]:
evaluator.plot_pr_curves()
plt.show()

In [ ]:
evaluator.plot_confusion_matrices()
plt.show()